In [0]:
CREATE OR REPLACE TABLE prata_passagens AS
WITH base AS (
    SELECT
        origem_url,
        destino_url,
        CAST(b.data_extracao AS TIMESTAMP) AS data_processamento,
        CAST(b.data_viagem AS TIMESTAMP) AS data_hora_partida,
        CAST(CAST(b.data_viagem AS TIMESTAMP) AS DATE) AS data_partida,
        origem,
        destino,
        COALESCE(
            get_json_object(b.raw_data, '$.routeConnectionId'),
            get_json_object(b.raw_data, '$.id'),
            concat(origem, destino, b.data_viagem, get_json_object(b.empresas_raw, '$[0]'))
        ) AS id_viagem,
        COALESCE(
            get_json_object(b.raw_data, '$.initialOriginStopDisplayName'),
            get_json_object(b.raw_data, '$.tripList[0].originStop.displayName')
        ) AS rodoviaria_origem,
        COALESCE(
            get_json_object(b.raw_data, '$.finalDestinationStopDisplayName'),
            get_json_object(b.raw_data, '$.tripList[0].destinationStop.displayName')
        ) AS rodoviaria_destino,
        get_json_object(b.raw_data, '$.totalDurationHour') AS duracao_viagem,
        CAST(get_json_object(b.raw_data, '$.totalTripMinutes') AS INT) AS duracao_minutos,
        get_json_object(b.empresas_raw, '$[0]') AS empresa,
        CAST(get_json_object(b.assentos_disponiveis_raw, '$[0]') AS INT) AS assentos_disponiveis,
        COALESCE(
            get_json_object(b.raw_data, '$.tripList[0].classFriendlyName'),
            get_json_object(b.raw_data, '$.serviceClass')
        ) AS classe_servico,
        CAST(b.preco AS DECIMAL(10, 2)) AS preco_passagem,
        DATEDIFF(
            CAST(CAST(b.data_viagem AS TIMESTAMP) AS DATE),
            CAST(CAST(b.data_extracao AS TIMESTAMP) AS DATE)
        ) AS dias_antecedencia,
        ROW_NUMBER() OVER (
            PARTITION BY 
                origem, 
                destino, 
                b.data_viagem, 
                get_json_object(b.empresas_raw, '$[0]'),
                COALESCE(get_json_object(b.raw_data, '$.tripList[0].classFriendlyName'), get_json_object(b.raw_data, '$.serviceClass'))
            ORDER BY b.data_extracao DESC
        ) AS rn
    FROM bronze_passagens b
    WHERE b.data_viagem IS NOT NULL
)
SELECT
    origem_url,
    destino_url,
    data_processamento,
    data_hora_partida,
    data_partida,
    origem,
    destino,
    rodoviaria_origem,
    rodoviaria_destino,
    duracao_viagem,
    duracao_minutos,
    empresa,
    assentos_disponiveis,
    classe_servico,
    preco_passagem,
    dias_antecedencia
FROM base
WHERE rn = 1
  AND (assentos_disponiveis > 0 OR assentos_disponiveis IS NULL);

SELECT * FROM prata_passagens LIMIT 10;

origem_url,destino_url,data_processamento,data_hora_partida,data_partida,origem,destino,rodoviaria_origem,rodoviaria_destino,duracao_viagem,duracao_minutos,empresa,assentos_disponiveis,classe_servico,preco_passagem,dias_antecedencia
belo-horizonte-mg-todos,rio-de-janeiro-rj-todos,2026-04-26T11:51:04.656Z,2026-04-26T13:20:00.000Z,2026-04-26,Belo Horizonte,Rio de Janeiro,"Terminal Turístico JK, Belo Horizonte - MG","Rodoviária Novo Rio, Rio de Janeiro - RJ",08:30:00,510,Util,7,SemiLeito,109.99,0
belo-horizonte-mg-todos,rio-de-janeiro-rj-todos,2026-04-26T11:51:04.656Z,2026-04-26T14:00:00.000Z,2026-04-26,Belo Horizonte,Rio de Janeiro,"Terminal Central, Belo Horizonte - MG","Rodoviária Novo Rio, Rio de Janeiro - RJ",08:01:00,481,Cometa,29,Convencional,219.99,0
belo-horizonte-mg-todos,rio-de-janeiro-rj-todos,2026-04-26T11:51:04.656Z,2026-04-26T16:00:00.000Z,2026-04-26,Belo Horizonte,Rio de Janeiro,"Terminal Central, Belo Horizonte - MG","Rodoviária Novo Rio, Rio de Janeiro - RJ",07:30:00,450,Cometa,6,SemiLeito,229.99,0
belo-horizonte-mg-todos,rio-de-janeiro-rj-todos,2026-04-26T11:51:04.656Z,2026-04-26T16:30:00.000Z,2026-04-26,Belo Horizonte,Rio de Janeiro,"Terminal Central, Belo Horizonte - MG","Rodoviária Novo Rio, Rio de Janeiro - RJ",07:20:00,440,Util,18,SemiLeito,219.99,0
belo-horizonte-mg-todos,rio-de-janeiro-rj-todos,2026-04-26T11:51:04.656Z,2026-04-26T18:00:00.000Z,2026-04-26,Belo Horizonte,Rio de Janeiro,"Terminal Central, Belo Horizonte - MG","Rodoviária Novo Rio, Rio de Janeiro - RJ",07:40:00,460,Util,2,Executivo,219.99,0
belo-horizonte-mg-todos,rio-de-janeiro-rj-todos,2026-04-26T11:51:04.656Z,2026-04-26T18:15:00.000Z,2026-04-26,Belo Horizonte,Rio de Janeiro,"Terminal Central, Belo Horizonte - MG","Rodoviária Novo Rio, Rio de Janeiro - RJ",07:55:00,475,Util,24,Executivo,219.99,0
belo-horizonte-mg-todos,rio-de-janeiro-rj-todos,2026-04-26T11:51:04.656Z,2026-04-26T18:30:00.000Z,2026-04-26,Belo Horizonte,Rio de Janeiro,"Terminal Central, Belo Horizonte - MG","Rodoviária Novo Rio, Rio de Janeiro - RJ",14:10:00,520,Auto Viação São Luiz,36,Convencional,226.55,0
belo-horizonte-mg-todos,rio-de-janeiro-rj-todos,2026-04-26T11:51:04.656Z,2026-04-26T19:30:00.000Z,2026-04-26,Belo Horizonte,Rio de Janeiro,"Terminal Central, Belo Horizonte - MG","Rodoviária Novo Rio, Rio de Janeiro - RJ",13:10:00,520,Auto Viação São Luiz,39,Convencional,226.55,0
belo-horizonte-mg-todos,rio-de-janeiro-rj-todos,2026-04-26T11:51:04.656Z,2026-04-26T19:45:00.000Z,2026-04-26,Belo Horizonte,Rio de Janeiro,"Terminal Central, Belo Horizonte - MG","Rodoviária Novo Rio, Rio de Janeiro - RJ",12:55:00,520,Auto Viação São Luiz,41,Convencional,226.55,0
belo-horizonte-mg-todos,rio-de-janeiro-rj-todos,2026-04-26T11:51:04.656Z,2026-04-26T20:45:00.000Z,2026-04-26,Belo Horizonte,Rio de Janeiro,"Terminal Central, Belo Horizonte - MG","Rodoviária Novo Rio, Rio de Janeiro - RJ",11:55:00,520,Auto Viação São Luiz,34,Convencional,226.55,0
